# 📚 Конспект: Система модулів та Стандартна бібліотека Python

> Урок 9 — повний конспект: система імпорту, пакети, `__name__`, `sys.path`,
> та 13 модулів стандартної бібліотеки з прикладами.

---

## 📋 Зміст

| # | Тема | Ключові концепції |
|---|------|-------------------|
| 1 | Навіщо існують модулі? | Організація, Простори імен, Перевикористання |
| 2 | Скрипти vs Модулі | `__name__ == "__main__"` |
| 3 | Що відбувається при `import`? | Search → Compile → Execute → Cache |
| 4 | `sys.path` та порядок пошуку | CWD → PYTHONPATH → stdlib → site-packages |
| 5 | Пакети та `__init__.py` | Структура пакету, стилі імпорту |
| 6 | Абсолютні vs Відносні імпорти | Порівняння, типові помилки |
| 7 | `datetime` | date, time, datetime, timedelta, strptime/strftime |
| 8 | `os` та `pathlib` | Файлова система, старий vs сучасний API |
| 9 | `math` | Константи, функції, `isclose` |
| 10 | `random` | Псевдовипадковість, seed, secrets |
| 11 | `re` | Регулярні вирази, групи захоплення |
| 12 | `collections` | Counter, defaultdict, deque |
| 13 | `itertools` | chain, product, combinations, takewhile |
| 14 | `functools` | lru_cache, partial, reduce |
| 15 | `json` | Серіалізація та десеріалізація |
| 16 | `timeit` | Бенчмаркінг продуктивності |
| 17 | `statistics` та `typing` | Статистика, підказки типів |
| 18 | Шпаргалка | Швидкий довідник |
| 19 | Задачі | Самостійна робота |

---

## 🧠 Ментальна модель: Python як конструктор

Python — мова «батарейки в комплекті» (batteries included).
Ядро мови маленьке, але разом зі стандартною бібліотекою (~200+ модулів)
воно утворює повноцінний конструктор для будь-яких задач.

```
Python Runtime
    ├── Вбудовані функції  (print, len, range — завжди доступні)
    ├── Стандартна бібліотека  (import math, datetime, re — завжди є)
    └── Сторонні пакети  (pip install requests — треба встановлювати)
```

Цей урок — про те, як Python знаходить і завантажує код,
і про найкорисніші модулі стандартної бібліотеки.

---

## 🏗️ Частина 1: Навіщо існують модулі?

### Що таке модуль?

**Будь-який `.py` файл — це модуль.** Ніякої магії.

```
my_project/
    main.py        ← модуль (можна імпортувати як 'main')
    utils.py       ← модуль (можна імпортувати як 'utils')
    math_tools.py  ← модуль (можна імпортувати як 'math_tools')
```

### Три причини, чому модулі існують

| # | Причина | Проблема без модулів |
|---|---------|---------------------|
| 1 | **Організація коду** | Один файл на 10 000 рядків — хаос |
| 2 | **Перевикористання** | Копіюєш функцію в кожен проект |
| 3 | **Простори імен** | Конфлікти імен між різними частинами |

### Найважливіше: Простори імен (Namespaces)

Простір імен — це ізольований «контейнер» для змінних і функцій.
Кожен модуль — окремий простір імен.

```python
# xml_reader.py
def parse(data):         # ← цей parse() живе в xml_reader
    return xml.parse(data)

# json_reader.py
def parse(data):         # ← цей parse() живе в json_reader
    return json.loads(data)

# main.py
import xml_reader
import json_reader

xml_reader.parse(...)    # ← чітко: яка саме parse()
json_reader.parse(...)   # ← чітко: яка саме parse()
# Конфлікту НЕМАЄ — різні простори імен!
```

> **Аналогія:** прізвища людей. У місті є 100 «Іванів», але
> «Іван Петренко» та «Іван Коваленко» — різні люди.
> Модуль — це «прізвище» для функцій.

---

## 🔀 Частина 2: Скрипти vs Модулі (`__name__ == "__main__"`)

### Ключове питання: як Python розрізняє «запускається напряму» vs «імпортується»?

Python автоматично встановлює спеціальну змінну `__name__`:

```
Запуск напряму:    python my_math.py    →  __name__ = '__main__'
Імпорт:           import my_math        →  __name__ = 'my_math'
```

### Навіщо це потрібно?

```python
# my_math.py
def add(a, b): return a + b

# ❌ БЕЗ перевірки __name__:
# Тести запускаються навіть при import my_math → небажано!
print(add(2, 3))    ← виконується при КОЖНОМУ import!

# ✅ З перевіркою __name__:
if __name__ == '__main__':
    print(add(2, 3))  ← виконується ТІЛЬКИ при прямому запуску
```

Це — стандартна конвенція Python. **Завжди** використовуй цей патерн
для тестового коду, `main()` функцій та точок входу.

In [ ]:
# my_math.py — демонстрація __name__
def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

# Цей блок виконується ТІЛЬКИ при запуску напряму
# При імпорті — ігнорується
if __name__ == "__main__":
    print(f"Тестуємо функції: add(2,3) = {add(2, 3)}")
    print(f"Тестуємо функції: multiply(4,5) = {multiply(4, 5)}")

# Перевіримо значення __name__ в нотебуці
print(f"__name__ = {__name__!r}")
# У нотебуці виведе '__main__' — бо ми запускаємо напряму
# При import my_math виведе 'my_math'

---

## ⚙️ Частина 3: Що відбувається при `import`? (4 кроки)

Коли Python зустрічає `import math`, він виконує чотири кроки:

```
┌─────────────────────────────────────────────────────────────┐
│                  import math — 4 кроки                      │
├──────────┬──────────────┬──────────────┬────────────────────┤
│ 1.SEARCH │  2. COMPILE  │  3. EXECUTE  │   4. CACHE         │
│          │              │              │                    │
│ Шукаємо  │ Компілюємо   │ Виконуємо    │ Зберігаємо в       │
│ math.py  │ у байткод    │ весь файл    │ sys.modules['math']│
│ за       │ .pyc файл    │ зверху вниз  │                    │
│ sys.path │              │              │ Повторний import → │
│          │              │              │ бере з кешу (швидко│
└──────────┴──────────────┴──────────────┴────────────────────┘
```

### Важливо: `sys.modules` — кеш завантажених модулів

Python **ніколи** не завантажує один модуль двічі.
Після першого `import math` результат зберігається у `sys.modules`.
Всі наступні `import math` просто беруть готовий об'єкт із кешу.

In [ ]:
import sys
import math  # стандартна бібліотека

# sys.modules — словник вже завантажених модулів
# Перевіримо чи math там є
print(f"'math' в sys.modules: {'math' in sys.modules}")
print(f"Де живе math: {sys.modules['math'].__file__}")
print()

# Важливо: повторний import НЕ перезапускає файл
import math  # Python просто бере з кешу sys.modules
print("Повторний import math — файл не перечитується!")
print(f"Об'єкт той самий: {id(sys.modules['math']) == id(math)}")
print()

# Скільки всього модулів вже в кеші?
print(f"Кількість модулів у sys.modules: {len(sys.modules)}")
# Переглянемо перші 5 ключів
first_five = list(sys.modules.keys())[:5]
print(f"Перші 5: {first_five}")

---

## 🗺️ Частина 4: `sys.path` та порядок пошуку модулів

### Де Python шукає модуль?

Коли ти пишеш `import mymodule`, Python переглядає директорії у `sys.path` **у порядку**:

```
1. Вбудовані модулі (built-ins: sys, builtins) — найвищий пріоритет
2. sys.modules кеш — чи вже завантажений?
3. sys.path[0] — поточна директорія (або директорія скрипту)
4. PYTHONPATH — змінна середовища (якщо встановлена)
5. Стандартна бібліотека — /usr/lib/python3.x/
6. site-packages — /usr/lib/python3.x/site-packages/ (pip install)
```

### ⚠️ Класична помилка: «Shadowing» (затінення)

Якщо ти назвеш свій файл `random.py` або `math.py`,
Python знайде **твій** файл першим і **не зможе завантажити** стандартний модуль!

```
❌ НЕ РОБИ ТАК:
my_project/
    random.py   ← «затіняє» стандартний random!
    math.py     ← «затіняє» стандартний math!
    os.py       ← «затіняє» стандартний os!

✅ ПРАВИЛЬНО:
my_project/
    my_random.py
    math_utils.py
    file_helpers.py
```

In [ ]:
import sys

print("Python шукає модулі в такому порядку:")
for i, path in enumerate(sys.path[:6], 1):
    print(f"  {i}. {path or '(поточна папка)'}")

print("\n⚠️ КЛАСИЧНА ПОМИЛКА — Shadowing:")
print("  Якщо ти створиш файл 'random.py' у своїй папці,")
print("  Python знайде ЙОГО, а не стандартний random!")
print()
print("  Діагностика: перевір звідки завантажився модуль")
import random
print(f"  random.__file__ = {random.__file__}")
print()
print("  Якщо виводить твій проект — ти маєш файл з такою назвою!")

---

## 📦 Частина 5: Пакети та `__init__.py`

### Що таке пакет?

**Пакет** — директорія з файлом `__init__.py` всередині.
Цей файл (може бути порожній) говорить Python: «це пакет, а не просто папка».

```
ecommerce/                ← пакет (є __init__.py)
    __init__.py           ← робить папку пакетом
    database.py           ← модуль ecommerce.database
    payments/             ← вкладений пакет
        __init__.py
        stripe.py         ← модуль ecommerce.payments.stripe
        paypal.py         ← модуль ecommerce.payments.paypal
    shipping/
        __init__.py
        nova_poshta.py
```

### Три стилі імпорту — порівняння

| Стиль | Синтаксис | Використання | Коли |
|-------|-----------|--------------|------|
| Модуль цілком | `import math` | `math.pi`, `math.sqrt()` | Багато функцій, потрібна ясність |
| Конкретні імена | `from math import pi, sqrt` | `pi`, `sqrt()` | Кілька функцій, часто використовуються |
| З псевдонімом | `from math import pi as π` | `π` | Довгі назви, уникнення конфліктів |

In [ ]:
# Різні способи імпорту — порівняння
import math                    # імпортуємо модуль цілком
from math import pi, sqrt      # імпортуємо конкретні імена
from math import pi as π       # імпортуємо з псевдонімом

print(f"math.pi   = {math.pi:.6f}")
print(f"pi        = {pi:.6f}")
print(f"π         = {π:.6f}")
print(f"sqrt(16)  = {sqrt(16)}")
print()

# Усі три способи дають той самий результат
print(f"Все те саме значення: {math.pi == pi == π}")
print()

# Перевіряємо атрибути модуля
print(f"math.__name__    = {math.__name__}")
print(f"math.__package__ = {math.__package__!r}")
# __file__ показує де фізично живе модуль
print(f"math.__file__    = {math.__file__}")

---

## ⚖️ Частина 6: Абсолютні vs Відносні імпорти

### Абсолютні імпорти — рекомендований стиль

```python
# Абсолютний — повний шлях від кореня пакету
from ecommerce.payments import stripe
from ecommerce.database import connect
```

### Відносні імпорти — тільки всередині пакету

```python
# Відносний — від поточного файлу
from . import database          # той самий пакет
from ..payments import stripe   # батьківський пакет
```

| | Абсолютний | Відносний |
|---|---|---|
| **Читабельність** | ✅ Зрозуміло звідки | ⚠️ Треба знати структуру |
| **Перенесення** | ✅ Не залежить від розташування | ❌ Ламається при переміщенні |
| **PEP 8** | ✅ Рекомендований | Допускається в пакетах |
| **Головний скрипт** | ✅ Працює | ❌ Не працює (`__main__`) |

### Типові помилки імпорту

```
❌ ПОМИЛКА 1: Wildcard import
   from math import *    ← забруднює namespace
   print(pi)             ← звідки? Незрозуміло!

❌ ПОМИЛКА 2: Circular import
   module_a.py: import module_b
   module_b.py: import module_a  ← КРАХ!

❌ ПОМИЛКА 3: Shadowing стандартних модулів
   Файл random.py → import random бере твій файл, не stdlib!
```

In [ ]:
# Демонстрація поширених помилок імпорту

# ❌ ПОМИЛКА 1: Wildcard import
# from math import *   ← забруднює namespace, незрозуміло звідки що
# print(pi)            ← хто такий pi? Треба шукати у 3 місцях!

# ✅ ПРАВИЛЬНО: явний імпорт
from math import pi, sqrt, ceil
print(f"Явний імпорт: pi={pi:.4f}, sqrt(9)={sqrt(9)}, ceil(3.2)={ceil(3.2)}")
print()

# ❌ ПОМИЛКА 2: Кільцевий імпорт (Circular import)
# module_a.py: import module_b
# module_b.py: import module_a  ← КРАХ! module_a ще не завантажений до кінця
print("Кільцевий імпорт — рішення:")
print("  1. Перенести спільну логіку в третій модуль module_c.py")
print("  2. Перемістити import всередину функції (lazy import)")
print()

# ✅ Lazy import — імпорт всередині функції (уникає circular imports)
def get_json_data(filepath):
    import json   # ← імпортується тільки при виклику функції
    with open(filepath) as f:
        return json.load(f)

print("Lazy import — функція визначена, але json ще не завантажений")
print(f"'json' в sys.modules до виклику: {'json' in __import__('sys').modules}")

---

# 🔋 ТЕМА 2: Стандартна бібліотека — «Батарейки в комплекті»

Python відомий своєю філософією **«batteries included»** — разом із мовою
поставляється величезна стандартна бібліотека, яка закриває більшість
повсякденних задач без жодних додаткових установок.

```
Стандартна бібліотека Python (~200+ модулів)
    ├── Математика:      math, statistics, decimal, fractions
    ├── Дати та час:     datetime, calendar, time
    ├── Файли та ОС:     os, pathlib, shutil, glob
    ├── Текст:           re, string, textwrap, difflib
    ├── Дані:            json, csv, pickle, sqlite3
    ├── Структури:       collections, heapq, bisect
    ├── Функції:         functools, itertools, operator
    ├── Тестування:      unittest, doctest
    ├── Мережа:          urllib, http, email
    └── Утиліти:         sys, logging, argparse, typing
```

> **Правило:** перед тим як шукати зовнішній пакет на PyPI,
> перевір чи немає потрібного модуля у стандартній бібліотеці!

---

## 🕐 Частина 7: `datetime` — Робота з часом

### Чотири головні класи модуля `datetime`

| Клас | Що зберігає | Приклад |
|------|------------|------|
| `date` | Лише дату (рік, місяць, день) | `date(2024, 10, 25)` |
| `time` | Лише час (год, хв, сек) | `time(14, 30, 55)` |
| `datetime` | Дату + час разом | `datetime(2024, 10, 25, 14, 30)` |
| `timedelta` | Різниця між двома моментами | `timedelta(days=7, hours=3)` |

### Мнемоніка strptime / strftime

```
str p time → str P arse time   → рядок → об'єкт datetime
str f time → str F ormat time  → об'єкт datetime → рядок
```

### Коди форматування

| Код | Значення | Приклад |
|-----|----------|---------|
| `%Y` | 4-цифровий рік | `2024` |
| `%m` | Місяць (01–12) | `10` |
| `%d` | День (01–31) | `25` |
| `%H` | Година 24h (00–23) | `14` |
| `%M` | Хвилина (00–59) | `30` |
| `%S` | Секунда (00–59) | `55` |
| `%B` | Повна назва місяця | `October` |
| `%A` | Повна назва дня | `Thursday` |

In [ ]:
from datetime import datetime, date, timedelta

# Поточний момент
now = datetime.now()
today = date.today()

print(f"Зараз:    {now}")
print(f"Сьогодні: {today}")
print(f"Рік:  {now.year}, Місяць: {now.month}, День: {now.day}")
print(f"Год:  {now.hour}, Хв:     {now.minute}")

In [ ]:
from datetime import datetime, timedelta

now = datetime.now()

# Арифметика дат за допомогою timedelta
ten_days_later  = now + timedelta(days=10)
two_weeks_ago   = now - timedelta(weeks=2)
in_90_days      = now + timedelta(days=90)

print(f"Зараз:           {now.strftime('%d.%m.%Y')}")
print(f"Через 10 днів:   {ten_days_later.strftime('%d.%m.%Y')}")
print(f"2 тижні тому:    {two_weeks_ago.strftime('%d.%m.%Y')}")
print(f"Через 90 днів:   {in_90_days.strftime('%d.%m.%Y')}")

# Різниця між датами
new_year = datetime(2025, 1, 1)
diff = now - new_year
print(f"\nДнів з початку 2025: {diff.days}")

In [ ]:
from datetime import datetime

# strPtime — Parse (рядок → об'єкт)
# strFtime — Format (об'єкт → рядок)

# 1. Парсинг рядка у datetime
date_str = "25.10.2023 14:30"
dt = datetime.strptime(date_str, "%d.%m.%Y %H:%M")
print(f"Рядок:   {date_str!r}")
print(f"Об'єкт:  {dt}")
print(f"Тип:     {type(dt)}")
print()

# 2. Форматування datetime у рядок
print(f"dd/mm/YYYY:       {dt.strftime('%d/%m/%Y')}")
print(f"Тільки час:       {dt.strftime('%H:%M')}")
print(f"Американський:    {dt.strftime('%m-%d-%Y')}")
print(f"Повний:           {dt.strftime('%d %B %Y, %H:%M')}")
print()

# 3. Повний цикл: рядок → об'єкт → +10 днів → рядок
input_str = "2023-09-20"
dt_obj = datetime.strptime(input_str, "%Y-%m-%d")
result = dt_obj + __import__('datetime').timedelta(days=10)
print(f"Вхід: {input_str}  →  +10 днів  →  Вихід: {result.strftime('%d %B %Y')}")

In [ ]:
from datetime import datetime

# ⚠️ ТИПОВА ПОМИЛКА: datetime.now() як default аргумент
# now() виконується ОДИН раз при завантаженні модуля — не при кожному виклику!

def log_bad(msg, timestamp=datetime.now()):   # ← ПОГАНО
    print(f"[{timestamp.strftime('%H:%M:%S')}] {msg}")

# ✅ ПРАВИЛЬНО: None як sentinel, now() всередині
def log_good(msg, timestamp=None):
    if timestamp is None:
        timestamp = datetime.now()            # ← обчислюється при кожному виклику
    print(f"[{timestamp.strftime('%H:%M:%S')}] {msg}")

import time
log_bad("Перший виклик")
time.sleep(0.1)
log_bad("Другий виклик")   # ← покаже той самий час!
print()
log_good("Перший виклик")
time.sleep(0.1)
log_good("Другий виклик")  # ← різний час ✓

---

## 📁 Частина 8: `os` та `pathlib` — Файлова система

### Старий підхід (`os`) vs Сучасний (`pathlib`)

| | `os` (старий) | `pathlib` (сучасний) |
|---|---|---|
| **Стиль** | Функції з рядками | Об'єктно-орієнтований |
| **З'єднання шляхів** | `os.path.join(a, b, c)` | `Path(a) / b / c` |
| **Читання файлу** | `open(os.path.join(...))` | `path.read_text()` |
| **Перевірка** | `os.path.exists(str)` | `path.exists()` |
| **Коли використовувати** | Змінні середовища, процеси | Всі операції з шляхами |

> **Рекомендація:** для роботи зі шляхами — завжди `pathlib`.
> `os` потрібен для `os.environ`, `os.getcwd()`, `os.listdir()`.

In [ ]:
import os

# Поточна директорія та змінні середовища
print(f"Поточна директорія: {os.getcwd()}")
print(f"HOME:  {os.environ.get('HOME', 'не знайдено')}")
print(f"PATH існує: {'PATH' in os.environ}")
print()

# Робота з файлами
print("Файли у поточній папці:")
for name in os.listdir('.'):
    if os.path.isfile(name):
        size = os.path.getsize(name)
        print(f"  📄 {name}: {size} байт")
    elif os.path.isdir(name):
        print(f"  📁 {name}/")

In [ ]:
from pathlib import Path

# pathlib — об'єктно-орієнтований підхід
# Оператор / для з'єднання шляхів (замість os.path.join)
p = Path('.')   # поточна папка

print(f"Поточний шлях: {p.resolve()}")
print(f"Домашня папка: {Path.home()}")
print()

# Побудова шляху через оператор /
data_path = Path('/tmp') / 'my_project' / 'data.csv'
print(f"Побудований шлях: {data_path}")
print(f"Батьківська папка: {data_path.parent}")
print(f"Ім'я файлу:        {data_path.name}")
print(f"Розширення:        {data_path.suffix}")
print(f"Ім'я без розш.:    {data_path.stem}")
print()

# Пошук файлів (glob)
print("*.ipynb файли:")
for nb in Path('.').glob('*.ipynb'):
    print(f"  {nb.name}")

---

## 🔢 Частина 9: `math` — Математика

Модуль `math` надає доступ до математичних функцій і констант.
Більшість функцій — обгортки над стандартними функціями C,
тому вони дуже швидкі.

> **⚠️ Важливо:** `math` працює тільки зі скалярними числами (`int`, `float`).
> Для масивів і матриць — використовуй `numpy`.

In [ ]:
import math

# Константи
print(f"π (pi):   {math.pi}")
print(f"e:        {math.e}")
print(f"τ (tau):  {math.tau}")
print()

# Основні функції
print(f"sqrt(16):    {math.sqrt(16)}")        # квадратний корінь
print(f"pow(2, 10):  {math.pow(2, 10)}")      # степінь
print(f"log(100,10): {math.log(100, 10)}")    # логарифм
print()

# Округлення
print(f"ceil(4.1):   {math.ceil(4.1)}")       # вгору
print(f"floor(4.9):  {math.floor(4.9)}")      # вниз
print(f"trunc(4.9):  {math.trunc(4.9)}")      # відкинути дробову
print()

# ⚠️ ВАЖЛИВО: порівняння float через isclose
a = 0.1 + 0.2
print(f"0.1 + 0.2 == 0.3:              {a == 0.3}")         # False через float!
print(f"math.isclose(0.1+0.2, 0.3):   {math.isclose(a, 0.3)}")  # True ✓

---

## 🎲 Частина 10: `random` — Псевдовипадковість

Модуль `random` генерує **псевдовипадкові** числа за допомогою алгоритму Mersenne Twister.

```
ПСЕВДО-випадковість:
    seed(42) → [результат завжди однаковий!]
    Не справжня випадковість — лише математична послідовність

⚠️ НЕ ВИКОРИСТОВУЙ для:
    - Паролів та токенів безпеки
    - Криптографії
    - UUID

✅ Використовуй для:
    - Тестових даних
    - Ігор та симуляцій
    - Вибірки для ML
    - Відтворюваних експериментів (з seed)

Для безпеки → import secrets
```

In [ ]:
import random

# Базові функції
print(f"randint(1, 6):    {random.randint(1, 6)}")      # як кубик (включно)
print(f"random():         {random.random():.4f}")         # 0.0 до 1.0
print(f"uniform(1.0,2.0): {random.uniform(1.0, 2.0):.4f}")
print()

# Робота з колекціями
fruits = ['яблуко', 'банан', 'вишня', 'диня', 'ківі']
print(f"choice():  {random.choice(fruits)}")                         # один елемент
print(f"sample(3): {random.sample(fruits, 3)}")                      # 3 без повтору
print()

# shuffle — перемішати на місці (нечиста!)
cards = ['Туз', 'Король', 'Дама', 'Валет', '10']
print(f"До shuffle:   {cards}")
random.shuffle(cards)
print(f"Після shuffle: {cards}")
print()

# seed — відтворюваність для тестів
random.seed(42)
print(f"seed(42) → randint: {[random.randint(1,100) for _ in range(5)]}")
random.seed(42)
print(f"seed(42) → randint: {[random.randint(1,100) for _ in range(5)]}")  # той самий!
print()
print("⚠️ Для паролів/токенів — тільки модуль 'secrets', не 'random'!")

---

## 🔍 Частина 11: `re` — Регулярні вирази

**Регулярний вираз (regex)** — шаблон для пошуку тексту.

### Головні функції

| Функція | Повертає | Використання |
|---------|----------|--------------|
| `re.search(pat, str)` | Перший збіг або `None` | Перевірити чи є |
| `re.findall(pat, str)` | Список усіх збігів | Знайти всі |
| `re.sub(pat, repl, str)` | Рядок із замінами | Замінити |
| `re.match(pat, str)` | Збіг на початку або `None` | Починається з |
| `re.compile(pat)` | Скомпільований об'єкт | Багаторазове використання |

### Найважливіші метасимволи

| Шаблон | Значення | Приклад |
|--------|----------|---------|
| `\d` | Цифра [0-9] | `\d{4}` — 4 цифри |
| `\w` | Буква або цифра | `\w+` — одне або більше |
| `\s` | Пробільний символ | `\s+` — пропуски |
| `.` | Будь-який символ | `.+` — будь-яке |
| `^` | Початок рядка | `^Hello` |
| `$` | Кінець рядка | `world$` |
| `(...)` | Група захоплення | `(\d{4})-(\d{2})` |

> **⚠️ Завжди використовуй `r''` (raw string) для regex!**
> `r'\d'` — правильно. `'\\d'` — те саме, але некрасиво.

In [ ]:
import re

text = "Контакти: Іван +380-97-123-4567, Марія +380-50-987-6543, email: ivan@example.com"

# re.findall — знайти всі збіги
phones = re.findall(r'\+380-\d{2}-\d{3}-\d{4}', text)
print(f"Телефони: {phones}")

# re.search — перший збіг (або None)
email_match = re.search(r'\w+@\w+\.\w+', text)
if email_match:
    print(f"Email: {email_match.group()}")
print()

# re.sub — заміна по шаблону
hidden = re.sub(r'\d', '*', text)
print(f"Прихований: {hidden}")
print()

# Групи захоплення — витягти частини
log = "2023-10-25 14:30:55 ERROR Connection timeout"
pattern = r'(\d{4}-\d{2}-\d{2}) (\d{2}:\d{2}:\d{2}) (\w+) (.+)'
m = re.match(pattern, log)
if m:
    print(f"Дата:    {m.group(1)}")
    print(f"Час:     {m.group(2)}")
    print(f"Рівень:  {m.group(3)}")
    print(f"Повідом: {m.group(4)}")
print()
print("⚠️ Завжди використовуй r'' (raw string) для регулярних виразів!")
print("   r'\\d' — правильно,  '\\\\d' — те саме, але некрасиво")

---

## 🗂️ Частина 12: `collections` — Розширені структури даних

Модуль `collections` містить спеціалізовані контейнери — розширення
стандартних `dict`, `list`, `tuple`.

### Три найважливіші класи

```
Counter     ← підраховує кількість елементів
             {елемент: кількість}

defaultdict ← dict, що автоматично створює значення за замовчуванням
             для відсутніх ключів — не потрібна перевірка if key not in d

deque       ← двостороння черга (double-ended queue)
             O(1) для append/pop з ОБОХ кінців
             (звичайний list: O(n) для appendleft/popleft)
```

In [ ]:
from collections import Counter, defaultdict, deque

# --- Counter: частотний аналіз ---
words = ['яблуко', 'банан', 'яблуко', 'вишня', 'банан', 'яблуко', 'диня']
counter = Counter(words)
print("Counter:")
print(f"  {counter}")
print(f"  most_common(2): {counter.most_common(2)}")
print(f"  'яблуко' зустрічається {counter['яблуко']} рази")
print()

# Counter для рядків
letter_freq = Counter("Привіт Харків")
print(f"Частота букв: {dict(letter_freq.most_common(5))}")
print()

# --- defaultdict: без перевірки ключа ---
# ❌ Звичайний dict — потребує перевірки
scores = {}
for student, score in [('Аня', 85), ('Боб', 92), ('Аня', 78), ('Боб', 88)]:
    if student not in scores:        # ← ручна перевірка
        scores[student] = []
    scores[student].append(score)
print(f"dict:        {scores}")

# ✅ defaultdict — автоматично створює порожній список
grouped = defaultdict(list)
for student, score in [('Аня', 85), ('Боб', 92), ('Аня', 78), ('Боб', 88)]:
    grouped[student].append(score)   # ← без перевірки!
print(f"defaultdict: {dict(grouped)}")
print()

# --- deque: двостороння черга ---
q = deque([1, 2, 3])
q.appendleft(0)    # O(1) — ефективно!
q.append(4)
print(f"deque: {q}")
print(f"popleft(): {q.popleft()}, deque: {q}")

---

## 🔗 Частина 13: `itertools` — Ітератори та пайплайни

Модуль `itertools` надає **лінивих** (lazy) ітераторів для ефективної
роботи з послідовностями. «Ліниві» — означає, що елементи обчислюються
**тільки коли потрібні**, без зберігання всіх даних у пам'яті.

```
itertools.chain(a, b)         ← з'єднати ітерабельні
itertools.islice(it, n)       ← взяти перші n елементів
itertools.product(a, b)       ← декартів добуток (всі пари)
itertools.combinations(a, n)  ← комбінації без повторів
itertools.takewhile(pred, it) ← брати поки pred повертає True
itertools.dropwhile(pred, it) ← пропускати поки pred True
```

In [ ]:
import itertools

# chain — об'єднати ітерабельні без копіювання в пам'ять
nums = [1, 2, 3]
letters = ['a', 'b', 'c']
combined = list(itertools.chain(nums, letters))
print(f"chain:         {combined}")

# islice — зріз ітератора (лінивий)
first_five = list(itertools.islice(range(1000000), 5))
print(f"islice(0..1M, 5): {first_five}")

# product — декартів добуток (всі комбінації)
sizes = ['S', 'M', 'L']
colors = ['Червоний', 'Синій']
products = list(itertools.product(sizes, colors))
print(f"product: {products}")

# combinations — без повторів
items = ['A', 'B', 'C', 'D']
combos = list(itertools.combinations(items, 2))
print(f"combinations(2): {combos}")

# takewhile / dropwhile
data = [2, 4, 6, 7, 8, 10]
took = list(itertools.takewhile(lambda x: x % 2 == 0, data))
print(f"takewhile(парні): {took}  ← зупиняється на першому непарному")

---

## ⚡ Частина 14: `functools` — Функції вищого порядку

Модуль `functools` надає інструменти для роботи з функціями як об'єктами.

```
lru_cache   ← Мемоізація: кешує результати функції
             Якщо вже викликали f(35) — повертає кешований результат
             LRU = Least Recently Used

partial     ← Часткове застосування: фіксує деякі аргументи
             square = partial(power, exponent=2)
             cube   = partial(power, exponent=3)

reduce      ← Редьюсер для довільних операцій
             reduce(operator.mul, [1,2,3,4,5]) → 120
```

In [ ]:
from functools import lru_cache, partial, reduce
import operator

# --- lru_cache: мемоізація (кешування результатів) ---
@lru_cache(maxsize=None)
def fib(n):
    """Числа Фібоначчі з кешуванням."""
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

import time
start = time.time()
result = fib(35)
elapsed = time.time() - start
print(f"fib(35) = {result}  (за {elapsed:.4f} сек)")
print(f"Кеш: {fib.cache_info()}")
print()

# --- partial: часткове застосування аргументів ---
def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)   # фіксуємо exponent=2
cube   = partial(power, exponent=3)   # фіксуємо exponent=3

print(f"square(5) = {square(5)}")
print(f"cube(3)   = {cube(3)}")
print()

# Практичний partial: налаштований print
from functools import partial
print_error = partial(print, "❌ ПОМИЛКА:", sep=' ')
print_ok    = partial(print, "✅ ОК:", sep=' ')
print_error("Файл не знайдено")
print_ok("Дані завантажено")
print()

# --- reduce: згортка (редьюсер для довільних операцій) ---
numbers = [1, 2, 3, 4, 5]
product = reduce(operator.mul, numbers)    # 1*2*3*4*5
print(f"reduce(mul, [1..5]) = {product}  (= 5!)")

---

## 📄 Частина 15: `json` — Серіалізація даних

JSON (JavaScript Object Notation) — найпоширеніший формат обміну даними.

### Відповідність типів Python ↔ JSON

| Python | JSON | Примітка |
|--------|------|---------|
| `dict` | `object` `{}` | |
| `list`, `tuple` | `array` `[]` | tuple → list при читанні |
| `str` | `string` | |
| `int`, `float` | `number` | |
| `True`/`False` | `true`/`false` | |
| `None` | `null` | |
| `datetime` | ❌ немає | Треба `.isoformat()` |

In [ ]:
import json

# Python dict → JSON рядок
data = {
    'name': 'Іван',
    'age': 28,
    'skills': ['Python', 'SQL'],
    'active': True,
    'score': 9.5
}

json_str = json.dumps(data, ensure_ascii=False, indent=2)
print("json.dumps() → рядок:")
print(json_str)
print()

# JSON рядок → Python dict
parsed = json.loads(json_str)
print(f"json.loads() → dict: {type(parsed)}")
print(f"  name: {parsed['name']}, skills: {parsed['skills']}")
print()

# Збереження у файл та читання
import tempfile, os
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False, encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
    fname = f.name

with open(fname, encoding='utf-8') as f:
    loaded = json.load(f)
print(f"Збережено та прочитано: {loaded['name']}, {loaded['age']} р.")
os.unlink(fname)
print()

# ⚠️ datetime не серіалізується напряму!
from datetime import datetime
try:
    json.dumps({'time': datetime.now()})
except TypeError as e:
    print(f"⚠️ TypeError: {e}")
    print("   Рішення: перетворити в рядок: datetime.now().isoformat()")

---

## ⏱️ Частина 16: `timeit` — Бенчмаркінг

Модуль `timeit` вимірює час виконання невеликих фрагментів коду.
Він автоматично запускає код **багато разів** і усереднює результат,
щоб зменшити вплив випадкових факторів.

```python
# Базовий синтаксис
timeit.timeit(stmt, setup='', number=1_000_000)
#              ↑ код     ↑ підготовка  ↑ кількість ітерацій

# Запуск з командного рядка
python -m timeit "''.join(str(i) for i in range(100))"
```

In [ ]:
import timeit

# Порівняємо list comprehension vs map
code1 = "[x**2 for x in range(100)]"
code2 = "list(map(lambda x: x**2, range(100)))"

t1 = timeit.timeit(code1, number=50_000)
t2 = timeit.timeit(code2, number=50_000)

print(f"List comprehension: {t1:.4f} сек")
print(f"map(lambda):        {t2:.4f} сек")
print(f"Переможець: {'List comprehension' if t1 < t2 else 'map'} швидший у {max(t1,t2)/min(t1,t2):.1f}x")
print()

# Порівняємо конкатенацію рядків
code3 = "result = ''; [result := result + str(i) for i in range(100)]"
code4 = "''.join(str(i) for i in range(100))"

t3 = timeit.timeit(code3, number=10_000)
t4 = timeit.timeit(code4, number=10_000)
print(f"Конкатенація +:     {t3:.4f} сек")
print(f"join():             {t4:.4f} сек")
print(f"join() швидший у {t3/t4:.1f}x разів")
print()
print("Запуск з командного рядка:")
print("  python -m timeit \"''.join(str(i) for i in range(100))\"")

---

## 📊 Частина 17: `statistics` та `typing`

### `statistics` — Статистичні функції

Модуль `statistics` надає функції для базового статистичного аналізу.
Для серйозної науки даних — `numpy` та `scipy`, але для простих задач
`statistics` з коробки вирішує 90% потреб.

### `typing` — Підказки типів

Підказки типів (type hints) — анотації, що документують які типи
очікує і повертає функція. **Python їх повністю ігнорує при виконанні** —
це лише документація для людей та IDE (PyCharm, VS Code).

```python
def greet(name: str, age: int) -> str:  ← type hints
    return f"{name}, {age}"             ← Python не перевіряє!

greet(123, "abc")   # Не помилка! Python дозволяє.
# Але IDE підсвітить як помилку — це і є ціль type hints.
```

In [ ]:
import statistics

grades = [85, 92, 78, 95, 88, 72, 91, 83, 79, 96]
print(f"Оцінки: {grades}")
print()
print(f"mean():   {statistics.mean(grades):.1f}")      # середнє
print(f"median(): {statistics.median(grades):.1f}")    # медіана
print(f"mode():   {statistics.mode(grades)}")          # мода
print(f"stdev():  {statistics.stdev(grades):.2f}")     # стандартне відхилення
print(f"variance:{statistics.variance(grades):.2f}")   # дисперсія
print()

# ⚠️ Медіана vs Середнє при викидах
salaries = [30000, 32000, 29000, 31000, 500000]   # 500k — директор
print(f"Зарплати: {salaries}")
print(f"mean:   {statistics.mean(salaries):,.0f}  ← спотворено викидом")
print(f"median: {statistics.median(salaries):,.0f}  ← реальніша картина")

In [ ]:
from typing import List, Dict, Tuple, Optional, Union

# Підказки типів — тільки для розробника та IDE
# Python їх ІГНОРУЄ при виконанні!

def process_scores(scores: List[int]) -> Dict[str, float]:
    """Аналізує список оцінок, повертає статистику."""
    return {
        "mean":   sum(scores) / len(scores),
        "max":    float(max(scores)),
        "min":    float(min(scores)),
    }

def greet(name: str, age: Optional[int] = None) -> str:
    """Optional[int] = int або None."""
    if age is not None:
        return f"Привіт, {name}! Тобі {age} років."
    return f"Привіт, {name}!"

def parse_value(value: Union[str, int]) -> int:
    """Union[str, int] = або рядок, або ціле число."""
    return int(value)

print(process_scores([85, 92, 78, 95, 88]))
print(greet("Іван"))
print(greet("Марія", 25))
print(parse_value("42"))
print(parse_value(42))
print()
print("Підказки типів НЕ впливають на виконання — це лише документація для людей та IDE")

---

## 📋 Частина 18: Шпаргалка — Система імпорту

```python
# Три стилі імпорту
import math                      # модуль цілком → math.pi
from math import pi, sqrt        # конкретні імена → pi
from math import pi as π         # з псевдонімом → π

# Перевірити звідки завантажився модуль
import random
print(random.__file__)

# Перевірити чи модуль у кеші
import sys
print('math' in sys.modules)

# sys.path — де Python шукає модулі
print(sys.path)
```

## 📋 Стандартна бібліотека — швидкий довідник

| Модуль | Задача | Ключові функції |
|--------|--------|-----------------|
| `datetime` | Робота з часом | `now()`, `strptime()`, `strftime()`, `timedelta` |
| `os` | ОС та файлова система | `getcwd()`, `listdir()`, `environ` |
| `pathlib` | Шляхи (ООП) | `Path()`, `/` оператор, `glob()` |
| `math` | Математика | `sqrt()`, `pi`, `ceil()`, `isclose()` |
| `random` | Псевдовипадковість | `randint()`, `choice()`, `shuffle()`, `seed()` |
| `re` | Регулярні вирази | `findall()`, `search()`, `sub()` |
| `collections` | Структури даних | `Counter`, `defaultdict`, `deque` |
| `itertools` | Ітератори | `chain()`, `product()`, `combinations()` |
| `functools` | Функції вищого порядку | `lru_cache`, `partial`, `reduce` |
| `json` | Серіалізація | `dumps()`, `loads()`, `dump()`, `load()` |
| `timeit` | Бенчмаркінг | `timeit()` |
| `statistics` | Статистика | `mean()`, `median()`, `stdev()` |
| `typing` | Підказки типів | `List`, `Dict`, `Optional`, `Union` |

---

# 📝 Частина 19: Задачі — Самостійна робота

### TODO 1: Аналіз тексту за допомогою `re` та `collections`

Дано текст з email-адресами різних доменів.
Знайдіть всі email-адреси та підрахуйте кількість кожного домену.

```
# Очікуваний вивід:
# Emails знайдено: 5
# Домени: Counter({'gmail.com': 2, 'example.com': 2, 'ukr.net': 1})
# Найпопулярніший: gmail.com (2 рази)
```

In [ ]:
import re
from collections import Counter

text = """
    Команда проекту:
    - Іван: ivan.petrov@gmail.com
    - Марія: maria@example.com
    - Олег: oleg.koval@ukr.net
    - Аня: anya@example.com
    - Боб: bob.smith@gmail.com
"""

# YOUR CODE HERE
# 1. Знайдіть всі email-адреси за допомогою re.findall
# emails = ...

# 2. Витягніть домени (частина після @)
# domains = ...

# 3. Підрахуйте частоту доменів за допомогою Counter
# domain_counts = ...

# 4. Виведіть результати
# print(f"Emails знайдено: {len(emails)}")
# print(f"Домени: {domain_counts}")
# print(f"Найпопулярніший: {domain_counts.most_common(1)[0][0]} ({domain_counts.most_common(1)[0][1]} рази)")

# Очікуваний вивід:
# Emails знайдено: 5
# Домени: Counter({'gmail.com': 2, 'example.com': 2, 'ukr.net': 1})
# Найпопулярніший: gmail.com (2 рази)

### TODO 2: Робота з датами

Дано список рядків з датами у форматі `'DD.MM.YYYY'`.
Для кожної дати:
1. Розпарсіть рядок у `datetime`
2. Додайте 30 днів
3. Відформатуйте результат як `'YYYY-MM-DD'`

```
# Очікуваний вивід:
# 01.09.2023 + 30 днів → 2023-10-01
# 15.11.2023 + 30 днів → 2023-12-15
# 20.12.2023 + 30 днів → 2024-01-19
```

In [ ]:
from datetime import datetime, timedelta

date_strings = ['01.09.2023', '15.11.2023', '20.12.2023']

# YOUR CODE HERE
# Для кожного рядка:
#   1. datetime.strptime(...)  ← рядок → об'єкт
#   2. + timedelta(days=30)    ← додати 30 днів
#   3. .strftime(...)          ← об'єкт → рядок формату YYYY-MM-DD

# for date_str in date_strings:
#     ...

# Очікуваний вивід:
# 01.09.2023 + 30 днів → 2023-10-01
# 15.11.2023 + 30 днів → 2023-12-15
# 20.12.2023 + 30 днів → 2024-01-19

### TODO 3: Бенчмаркінг

Порівняйте два підходи до перевірки чи число є простим:
- **Наївний:** перебираємо всі дільники від 2 до n
- **Оптимізований:** перебираємо тільки до `sqrt(n)`

Виміряйте час обох підходів для n=10_000 за допомогою `timeit`.

```
# Очікуваний вивід (приблизно):
# Наївний:        X.XXXX сек
# Оптимізований: X.XXXX сек
# Прискорення: X.Xx
```

In [ ]:
import timeit
import math

# Наївний алгоритм — O(n)
def is_prime_naive(n):
    if n < 2:
        return False
    # YOUR CODE HERE: перебрати від 2 до n-1
    pass

# Оптимізований — O(sqrt(n))
def is_prime_fast(n):
    if n < 2:
        return False
    # YOUR CODE HERE: перебрати від 2 до int(math.sqrt(n)) + 1
    pass

# Перевірка коректності (повинні дати однакові результати)
# test_nums = [2, 3, 4, 17, 100, 97, 1000003]
# for n in test_nums:
#     assert is_prime_naive(n) == is_prime_fast(n), f"Різні результати для {n}!"
# print("Обидві функції дають однаковий результат ✓")

# Бенчмаркінг
# setup = 'from __main__ import is_prime_naive, is_prime_fast'
# t1 = timeit.timeit('is_prime_naive(10_000)', setup=setup, number=10_000)
# t2 = timeit.timeit('is_prime_fast(10_000)',  setup=setup, number=10_000)
# print(f"Наївний:        {t1:.4f} сек")
# print(f"Оптимізований: {t2:.4f} сек")
# print(f"Прискорення: {t1/t2:.1f}x")

### TODO 4: Пайплайн з `itertools` та `collections`

Дано список слів різної довжини.
Побудуйте пайплайн:
1. Залишіть тільки слова довжиною > 4 символів
2. `itertools.chain` — з'єднайте всі букви цих слів в один потік
3. `Counter` — підрахуйте частоту кожної букви (без пробілів, lowercase)
4. Виведіть топ-5 найпоширеніших букв

```
# Очікуваний вивід (приблизно):
# Слова > 4 літери: ['python', 'programming', 'функція', 'модуль', 'бібліотека']
# Топ-5 букв: [('о', 3), ('і', 3), ('а', 2), ('н', 2), ('г', 2)]
```

In [ ]:
import itertools
from collections import Counter

words = ['if', 'python', 'or', 'programming', 'функція', 'and', 'модуль', 'бібліотека', 'for']

# YOUR CODE HERE
# 1. Відфільтруйте слова довжиною > 4
# long_words = ...

# 2. itertools.chain — з'єднайте всі букви в один ітератор
# all_chars = itertools.chain(*long_words)

# 3. Counter — підрахуйте букви (lowercase, без пробілів)
# letter_counts = Counter(ch.lower() for ch in all_chars if ch.strip())

# 4. Виведіть топ-5
# print(f"Слова > 4 літери: {long_words}")
# print(f"Топ-5 букв: {letter_counts.most_common(5)}")

### TODO 5 ⭐: Мемоізація вручну

Реалізуйте кешування результатів функції Фібоначчі **без** використання
`lru_cache` — власний словник-кеш.

Потім порівняйте швидкість трьох підходів за допомогою `timeit`:
1. Наївна рекурсія (без кешу)
2. Ваша реалізація з dict-кешем
3. З `@lru_cache`

```
# Очікуваний вивід (приблизно):
# fib_naive(30) = 832040
# fib_cached(30) = 832040
# fib_lru(30) = 832040
#
# Наївна:      X.XXXX сек
# Dict-кеш:    X.XXXX сек
# lru_cache:   X.XXXX сек
# Прискорення кешу: ~XXXx
```

In [ ]:
import timeit
from functools import lru_cache

# 1. Наївна рекурсія — без кешу
def fib_naive(n):
    if n < 2:
        return n
    return fib_naive(n-1) + fib_naive(n-2)

# 2. YOUR CODE HERE — з власним dict-кешем
_cache = {}   # словник для зберігання результатів

def fib_cached(n):
    # Підказка: спочатку перевір чи є n у _cache
    # якщо є — поверни з кешу
    # якщо немає — обчисли, збережи в _cache, поверни
    pass

# 3. З lru_cache
@lru_cache(maxsize=None)
def fib_lru(n):
    if n < 2:
        return n
    return fib_lru(n-1) + fib_lru(n-2)

# Перевірка коректності
# print(f"fib_naive(30)  = {fib_naive(30)}")
# print(f"fib_cached(30) = {fib_cached(30)}")
# print(f"fib_lru(30)    = {fib_lru(30)}")

# Бенчмаркінг (попередження: fib_naive дуже повільний для n=35!)
# setup = 'from __main__ import fib_naive, fib_cached, fib_lru; fib_lru.cache_clear()'
# t1 = timeit.timeit('fib_naive(28)',  setup=setup, number=10)
# t2 = timeit.timeit('fib_cached(28)', setup=setup, number=1000)
# t3 = timeit.timeit('fib_lru(28)',    setup=setup, number=1000)
# print(f"\nНаївна:      {t1:.4f} сек (10 разів)")
# print(f"Dict-кеш:    {t2:.4f} сек (1000 разів)")
# print(f"lru_cache:   {t3:.4f} сек (1000 разів)")